# SQL Querying with SQLite in Python

**Skills demonstrated:** DDL (`CREATE TABLE`), DML (`INSERT`, `DELETE`), `SELECT` with `WHERE`, `LIKE` pattern matching, range filters, `JOIN` with aliases and computed columns, nested correlated subqueries

**Tools:** SQLite (in-memory), `ipython-sql`, `prettytable`, Google Colab

---

## Scenario

A company stores employee and manager records in two related tables linked by `dept_id`. This notebook builds both tables in an in-memory SQLite database, inserts sample data, and answers five business questions using progressively complex SQL.

---

## Database Schema

### `employees`
| Column | Type | Description |
|--------|------|-------------|
| `employee_num` | INTEGER | Unique employee ID |
| `last_name` | TEXT | Last name |
| `first_name` | TEXT | First name |
| `salary` | INTEGER | Annual salary (USD) |
| `dept_id` | INTEGER | Department ID — links to `managers.dept_id` |
| `age` | INTEGER | Age in years |

### `managers`
| Column | Type | Description |
|--------|------|-------------|
| `mgr_num` | INTEGER | Unique manager ID |
| `last_name` | TEXT | Last name |
| `first_name` | TEXT | First name |
| `salary` | INTEGER | Annual salary (USD) |
| `dept_id` | INTEGER | Department ID — links to `employees.dept_id` |
| `age` | INTEGER | Age in years |

## 1. Setup

`sqlite://` with no file path creates a temporary in-memory database. `ipython-sql` enables the `%%sql` magic command for writing SQL directly in notebook cells.

In [80]:
!pip install -q prettytable==3.10.0 ipython-sql==0.5.0

%load_ext sql
%sql sqlite://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


---
## 2. Create Tables (DDL)

`CREATE TABLE IF NOT EXISTS` is a safe, idempotent pattern — the notebook can be re-run from the top without raising errors if the tables already exist.

In [81]:
%%sql

CREATE TABLE IF NOT EXISTS employees (
    employee_num INTEGER,
    last_name    TEXT,
    first_name   TEXT,
    salary       INTEGER,
    dept_id      INTEGER,
    age          INTEGER
);

 * sqlite://
Done.


In [82]:
%%sql

CREATE TABLE IF NOT EXISTS managers (
    mgr_num    INTEGER,
    last_name  TEXT,
    first_name TEXT,
    salary     INTEGER,
    dept_id    INTEGER,
    age        INTEGER
);

 * sqlite://
Done.


---
## 3. Insert Sample Data (DML)

The `DELETE` before inserting makes the notebook idempotent — re-running will not create duplicate rows.

In [83]:
%%sql
DELETE FROM employees;

 * sqlite://
32 rows affected.


In [84]:
%%sql

-- 4 employees across 2 departments (dept_id 500 and 501)
INSERT INTO employees VALUES (1001, 'Smith',    'John',  62000, 500, 32);
INSERT INTO employees VALUES (1002, 'Anderson', 'Jane',  87500, 500, 40);
INSERT INTO employees VALUES (1003, 'Everest',  'Brad',  71000, 501, 26);
INSERT INTO employees VALUES (1004, 'Horvath',  'Brain', 62000, 501, 44);

 * sqlite://
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.


In [85]:
%%sql

-- 3 managers: White manages dept 500, Smith manages dept 501, Cowgill manages dept 502
INSERT INTO managers VALUES (101, 'White',   'Jonathan', 152000, 500, 38);
INSERT INTO managers VALUES (102, 'Smith',   'Paul',     120000, 501, 40);
INSERT INTO managers VALUES (103, 'Cowgill', 'Richard',  150000, 502, 50);

 * sqlite://
1 rows affected.
1 rows affected.
1 rows affected.


In [86]:
%config SqlMagic.style = "DEFAULT"

---
## 4. Queries

Five queries covering the SQL concepts most commonly tested in data analytics interviews and used in production analytics work.

### Query 1 — Employees older than 30

A `WHERE` filter on a numeric column. In analytics, this pattern is the foundation of demographic segmentation — filtering customers, employees, or users by age, tenure, or any numeric attribute.

**Expected output:** `last_name`, `first_name`, `age` where `age > 30`

In [87]:
%%sql
SELECT last_name, first_name, age
FROM employees
WHERE age > 30;

  last_name first_name  age
0     Smith       John   32
1  Anderson       Jane   40
2   Horvath      Brain   44

**Result:** Smith (32), Anderson (40), and Horvath (44) qualify. Everest (26) is excluded.

### Query 2 — Employees whose first name ends in "n"

`LIKE` with a `%` wildcard matches any sequence of characters. `"%n"` means: any string ending with the letter *n*. This is equivalent to Python's `str.endswith('n')` or the regex pattern `n$`.

**Expected output:** `last_name`, `first_name` where `first_name` ends in "n"

In [88]:
%%sql
SELECT last_name, first_name
FROM employees
WHERE first_name LIKE "%n";

  last_name first_name
0     Smith       John
1   Horvath      Brain

**Result:** "John" and "Brain" both end in *n*. "Jane" ends in *e* and "Brad" ends in *d* — neither matches.

### Query 3 — Managers earning between $130k and $160k

A range filter on the `managers` table using two `AND`-connected conditions. Written explicitly (rather than `BETWEEN`) to make the exclusive boundary logic clear — values exactly equal to $130k or $160k would not be returned.

**Expected output:** `last_name`, `first_name`, `salary` where `130000 < salary < 160000`

In [89]:
%%sql
SELECT last_name, first_name, salary
FROM managers
WHERE salary > 130000
  AND salary < 160000;

  last_name first_name  salary
0     White   Jonathan  152000
1   Cowgill    Richard  150000

**Result:** White ($152k) and Cowgill ($150k) fall within the range. Smith ($120k) is below the lower bound.

### Query 4 — Employees older than their manager

This query joins `employees` and `managers` on `dept_id` and computes a derived column `ageDiff` showing how many years older the employee is than their manager. Table aliases (`e`, `m`) keep the query concise and readable.

This is a core cross-table comparison pattern used in HR analytics, performance reporting, and any scenario requiring row-level comparisons between two related entities.

**Expected output:** employee name, manager name, `ageDiff` — only where employee is older

In [90]:
%%sql

-- JOIN on dept_id links each employee to their department's manager
-- ageDiff is a computed column: positive value means employee is older than manager
SELECT
    e.last_name  AS emp_LastName,
    e.first_name AS emp_FirstName,
    m.last_name  AS mgr_LastName,
    m.first_name AS mgr_FirstName,
    e.age - m.age AS ageDiff
FROM employees e
JOIN managers m
    ON e.dept_id = m.dept_id
WHERE e.age > m.age;

  emp_LastName emp_FirstName mgr_LastName mgr_FirstName  ageDiff
0     Anderson          Jane        White      Jonathan        2
1      Horvath         Brain        Smith          Paul        4

**Result:**
- Anderson (40) is 2 years older than her manager White (38) in dept 500
- Horvath (44) is 4 years older than his manager Smith (40) in dept 501

Smith (32) and Everest (26) are younger than their respective managers and are excluded.

### Query 5 — Employee with the 2nd highest salary

This uses a **nested correlated subquery** — three levels of `SELECT` working inward to outward:

| Level | What it does | Result |
|-------|-------------|--------|
| Innermost | `MAX(salary)` from all employees | $87,500 (Anderson) |
| Middle | `MAX(salary)` excluding the top value | $71,000 |
| Outer | Returns employees whose salary equals that value | Brad Everest |

This approach works in any SQL dialect including SQLite, which does not support window functions like `RANK()` or `DENSE_RANK()`.

In [91]:
%%sql

SELECT *
FROM employees
WHERE salary = (
    SELECT MAX(salary)
    FROM employees
    WHERE salary != (SELECT MAX(salary) FROM employees)  -- exclude the top salary
);

   employee_num last_name first_name  salary  dept_id  age
0          1003   Everest       Brad   71000      501   26

**Result:** Brad Everest ($71,000) holds the 2nd highest salary.

Full salary ranking:
1. Jane Anderson — $87,500
2. **Brad Everest — $71,000** ← returned
3. John Smith / Brain Horvath — $62,000 (tied)

---
## Conclusion

| Query | SQL Concept | Business Use Case |
|-------|------------|------------------|
| 1 | `WHERE` numeric filter | Demographic segmentation |
| 2 | `LIKE` with `%` wildcard | Pattern-based text search |
| 3 | Range filter with `AND` | Salary band analysis |
| 4 | `JOIN` + computed column | Cross-table comparison, HR analytics |
| 5 | Nested correlated subquery | Rank-based record retrieval |

**Skills demonstrated:** relational schema design, DDL and DML, single-table filtering, text pattern matching, range conditions, multi-table joins with table aliases and derived columns, and nested subquery logic — the core SQL skill set for data analytics and data engineering roles.